# 使用 @tool 装饰器(推荐)
## 1. 自定义工具描述：description
函数使用 @tool 修饰以后就是一个可以被模型识别的工具了
如果没有提供 description 参数的情况下，要求函数必须提供 docstring。否则不用。
当同时使用了两者的情况下，description 优先级更高

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from traitlets.utils import descriptions


# 会报错，因为用了 @tool，但是没有提供 description，也没有提供 docstring
@tool
def get_weather(city: str):
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool

# docstring
@tool
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool

# description
@tool(description = "天气查询工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool

# 当我们没有向 @tool 传递 description 参数时，默认情况下， tool 会将 docstring 整体视为 description （city 无法识别），如下
@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市
    """
    return f"{city}天气晴朗"

# 当我们没有向 @tool 传递 description 参数时，通过将 parse_docstring 设置为True，docstring会被解析，填充到相应的字段描述中
@tool(parse_docstring=True)
def get_weather1(city: str):
    """
    天气查询工具

    Args:
        city: 城市
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

## 2. 更改工具名称：name_or_callable

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool

@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

## 3. 自定义 args_schema
### 3.1 使用 Pydantic 模型定义
通过继承核心基类 BaseModel 定义数据模型，从而声明字段结构、类型约束、默认值以0及校验规则。

In [ ]:
from pydantic import BaseModel, Filed

class WeatherInput(BaseModel):
    city: str
    nature: str = Filed(
        description="国家",
        default="中国"  # 默认值
    )

print(WeatherInput(city="北京", nature="中国"))

@tool(args_schema=WeatherInput)
def get_weather(city: str, nature: str):
    """
    天气查询工具
    """
    return f"{nature}-{city}天气晴朗"

print(convert_to_openai_tool(get_weather))

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京",
        description="城市"
    )
    # 使用 Literal类型限定参数为固定选项,表示字段不能是任意某种类型的值，而只能是几个固定字面量之一
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )

print(WeatherInput())

## 3.2 使用 json_schema 形式

In [ ]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

# {
#     "type": "function",
#     "function": {
#         "name": "get_weather",
#         "description": "获取当日天气，可选未来五日天气预报",
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "location": {"type": "string"},
#                 "units": {"type": "string"},
#                 "include_forecast": {"type": "boolean"}
#             },
#             "required": ["location", "units", "include_forecast"]
#         },
#     }
# }

weather_schema = {
    "properties": {
        "city": {"default": "beijing", "description": "具体城市", "type": "string"},
        "units": {"enum": ["celsius", "fahrenheit"], "type": "string"},
        "include_forecast": {"type": "boolean", "default": False, "description": "是否包含未来五天天气预报"}
    },
    "type": "object",
    "required": ["location", "units", "include_forecast"]
}

@tool(args_schema=weather_schema)
def get_weather(city: str, units: str = "celsius", include_forecast: bool = False):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get_weather))